# 星巴克订单探索性数据分析

本项目分析 10 万条星巴克订单，探索订单特征、顾客特征与满意度之间的关系。内容包括数据加载与初步认识、探索性数据分析，以及一段统计分析拓展：描述性统计、异常值检查、相关性分析和统计检验。

参考并改编自 Kaggle Notebook：[Starbucks Orders: What Your Coffee Says About You](https://www.kaggle.com/code/seki32/starbucks-orders-what-your-coffee-says-about-you)。本版本聚焦探索分析，不包含特征工程和预测模型。

数据集：[Starbucks Customer Ordering Patterns](https://www.kaggle.com/datasets/likithagedipudi/starbucks-customer-ordering-patterns)。在 Kaggle 中运行时，请先将数据集添加为 Notebook 输入。

## 1. 数据加载与初步认识

先读取订单数据，了解数据规模、字段类型和基本统计特征。每一行代表一笔订单。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import shapiro, chi2_contingency, kruskal
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Noto Sans CJK SC', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

print('分析环境已就绪。')

In [ ]:
from pathlib import Path

kaggle_path = Path('/kaggle/input/datasets/likithagedipudi/starbucks-customer-ordering-patterns/starbucks_customer_ordering_patterns.csv')
local_path = Path('starbucks_customer_ordering_patterns.csv')
downloads_path = Path.home() / 'Downloads' / 'starbucks_customer_ordering_patterns.csv'
data_path = next((p for p in (kaggle_path, local_path, downloads_path) if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("找不到数据集。请在 Kaggle 添加数据集，或将 CSV 放在 Notebook 同目录或 Downloads 文件夹。")
df = pd.read_csv(data_path)

print(f"数据规模：{df.shape[0]:,} 行 × {df.shape[1]} 列")
print(f"内存占用：{df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print()
print('字段类型：')
print(df.dtypes.to_string())

In [ ]:
df.head(10)

In [ ]:
df.describe().round(2)

初步统计显示，数据包含约 10 万笔订单和约 1.5 万位顾客，因此不少顾客有多次消费记录。平均每笔订单约有 3.7 件商品、约 2 项定制，消费约 14.5 美元，出单时间约 7 分钟；满意度均值约为 3.69 分（满分 5 分）。

接下来通过分布图和分组比较，进一步检查这些特征与满意度之间的关系。

## 2. 探索性数据分析（EDA）

### 2.1 缺失值分析

In [ ]:
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)

missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0]

if len(missing_df) == 0:
    print("Zero missing values across all 20 columns.")
    print("This dataset is squeaky clean. No imputation needed, no rows to drop.")
    print("That is honestly rare. Let's appreciate it and move on.")
else:
    print(missing_df.sort_values('Missing %', ascending=False))

数据集中没有缺失值，因此无需填补或删除缺失记录。

### 2.2 满意度分布

目标变量 `customer_satisfaction` 的取值为 1 至 5 分。先查看各评分的数量和比例，确认分布是否均衡。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

counts = df['customer_satisfaction'].value_counts().sort_index()
colors = ['#c0392b', '#e67e22', '#f1c40f', '#27ae60', '#2ecc71']

axes[0].bar(counts.index, counts.values, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_xlabel('满意度评分')
axes[0].set_ylabel('Count')
axes[0].set_title('顾客满意度分布')
for i, (idx, val) in enumerate(zip(counts.index, counts.values)):
    axes[0].text(idx, val + 500, f'{val:,}\n({val/len(df)*100:.1f}%)',
                ha='center', fontsize=10, fontweight='bold')

cumulative = counts.cumsum() / len(df) * 100
axes[1].plot(counts.index, cumulative.values, 'o-', color='#2c3e50', linewidth=2, markersize=8)
axes[1].axhline(y=50, color='#e74c3c', linestyle='--', alpha=0.7, label='50%参考线')
axes[1].set_xlabel('满意度评分')
axes[1].set_ylabel('累计百分比')
axes[1].set_title('累计占比')
axes[1].legend()
axes[1].set_ylim(0, 105)

plt.tight_layout()
plt.show()

print(f"平均满意度: {df['customer_satisfaction'].mean():.3f}")
print(f"满意度中位数: {df['customer_satisfaction'].median()}")
print(f"Mode: {df['customer_satisfaction'].mode()[0]}")

![图表：01-satisfaction-distribution.png](images/01-satisfaction-distribution.png)

满意度评分整体偏向较高分：评分 4 和 5 的顾客占多数，评分 1 的比例较低；最常见的评分是 4。后续比较分组时需要留意各评分组的样本数量差异。

### 2.3 数值特征分布

![图表：02-numeric-distributions.png](images/02-numeric-distributions.png)

In [ ]:
numerical_cols = ['cart_size', 'num_customizations', 'total_spend', 'fulfillment_time_min']

fig, axes = plt.subplots(2, 4, figsize=(18, 10))

for i, col in enumerate(numerical_cols):
    axes[0, i].hist(df[col], bins=40, color='#3498db', edgecolor='white', alpha=0.85)
    axes[0, i].set_title(f'{col}')
    axes[0, i].axvline(df[col].mean(), color='#e74c3c', linestyle='--', label=f'均值： {df[col].mean():.1f}')
    axes[0, i].axvline(df[col].median(), color='#2ecc71', linestyle='--', label=f'Median: {df[col].median():.1f}')
    axes[0, i].legend(fontsize=8)

    sns.boxplot(x=df[col], ax=axes[1, i], color='#3498db')
    axes[1, i].set_title(f'{col} （箱线图）')

plt.suptitle('数值特征分布', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

图中展示了购物车商品数、定制数量、消费金额和履约时间的直方图及箱线图。总体来看，购物车和定制数量多数处于较低范围；消费金额会随商品数量变化；履约时间分布较广。异常值将在统计拓展部分进一步检查。

![图表：03-categorical-distributions.png](images/03-categorical-distributions.png)

### 2.4 类别特征分布

In [ ]:
cat_cols = ['order_channel', 'store_location_type', 'region',
            'customer_age_group', 'customer_gender', 'drink_category', 'day_of_week']

fig, axes = plt.subplots(3, 3, figsize=(18, 16))
axes = axes.flatten()

palette = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c', '#e67e22']

for i, col in enumerate(cat_cols):
    order = df[col].value_counts().index
    sns.countplot(data=df, x=col, ax=axes[i], order=order,
                  palette=palette[:len(order)], edgecolor='white')
    axes[i].set_title(f'{col}', fontweight='bold')
    axes[i].tick_params(axis='x', rotation=45)
    total = len(df)
    for p in axes[i].patches:
        pct = f'{p.get_height()/total*100:.1f}%'
        axes[i].annotate(pct, (p.get_x() + p.get_width()/2., p.get_height()),
                        ha='center', va='bottom', fontsize=8)

for j in range(len(cat_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('类别特征分布', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

类别特征包括下单渠道、门店位置、地区、年龄组、性别、饮品类别和星期。图中展示了各类别的订单数量和占比，可用于检查样本覆盖情况。不同类别的占比若不均衡，解释后续分组结果时应考虑样本量。

![图表：04-numeric-features-vs-satisfaction.png](images/04-numeric-features-vs-satisfaction.png)

### 2.5 数值特征与满意度

![图表：05-categorical-features-vs-satisfaction.png](images/05-categorical-features-vs-satisfaction.png)

将数值特征按满意度分组，比较其分布和均值。箱线图可以帮助观察不同评分组之间的差异及组内波动。

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for i, col in enumerate(numerical_cols):
    row, c = divmod(i, 2)
    sns.boxplot(data=df, x='customer_satisfaction', y=col, ax=axes[row, c],
                palette='RdYlGn')
    means = df.groupby('customer_satisfaction')[col].mean()
    axes[row, c].plot(means.index, means.values, 'D-', color='black',
                      markersize=6, linewidth=1.5, label='Mean')
    axes[row, c].set_title(f'{col} vs Satisfaction', fontweight='bold')
    axes[row, c].legend()

plt.suptitle('数值特征与满意度', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("各满意度评分组的特征均值：")
print(df.groupby('customer_satisfaction')[numerical_cols].mean().round(2).to_string())

箱线图显示，不同满意度组的购物车商品数、定制数量和消费金额分布有较多重叠；履约时间随满意度变化的趋势相对明显。图表描述的是关联，不能据此判断履约时间变化导致了满意度变化。

![图表：06-boolean-features-vs-satisfaction.png](images/06-boolean-features-vs-satisfaction.png)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

cat_interest = ['order_channel', 'store_location_type', 'customer_age_group', 'drink_category']

for i, col in enumerate(cat_interest):
    row, c = divmod(i, 2)
    ct = pd.crosstab(df[col], df['customer_satisfaction'], normalize='index') * 100
    ct.plot(kind='bar', stacked=True, ax=axes[row, c], colormap='RdYlGn', edgecolor='white')
    axes[row, c].set_title(f'{col} vs Satisfaction', fontweight='bold')
    axes[row, c].set_ylabel('百分比')
    axes[row, c].tick_params(axis='x', rotation=45)
    axes[row, c].legend(title='Score', bbox_to_anchor=(1.0, 1.0), fontsize=8)

plt.suptitle('类别特征与满意度', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
print("平均满意度 by categorical features:\n")
for col in cat_cols:
    means = df.groupby(col)['customer_satisfaction'].mean().sort_values(ascending=False)
    print(f"{col}:")
    for idx, val in means.items():
        print(f"  {idx}: {val:.3f}")
    print()

类别特征分组结果可用于比较不同渠道、门店位置、年龄组和饮品类别下的评分构成。整体差异如较小，应结合样本数量与效应大小判断其实际意义。履约时间仍是前述图表中较值得进一步检验的变量。

### 2.6 布尔特征与满意度

In [ ]:
bool_cols = ['is_rewards_member', 'has_food_item', 'order_ahead']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, col in enumerate(bool_cols):
    means = df.groupby(col)['customer_satisfaction'].mean()
    bars = axes[i].bar([str(x) for x in means.index], means.values,
                       color=['#e74c3c', '#2ecc71'], edgecolor='white', linewidth=1.5)
    axes[i].set_title(f'平均满意度： {col}', fontweight='bold')
    axes[i].set_ylabel('平均满意度')
    axes[i].set_ylim(3.0, 4.5)
    for bar, val in zip(bars, means.values):
        axes[i].text(bar.get_x() + bar.get_width()/2., val + 0.02,
                    f'{val:.3f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

分别比较会员身份、订单是否含食品、是否提前下单等布尔特征对应的平均满意度。若组间存在差异，它们可以作为进一步分析的线索；但观察性数据不能说明会员身份或提前下单本身造成了满意度变化。

### 2.7 统计分析拓展

前面的图表帮助我们发现模式。这里用描述统计、相关系数和统计检验作补充。样本量很大时，即使差异很小也可能得到较小的 p 值，因此需要结合效应大小和实际意义解读。统计关联不代表因果关系。

#### 2.7.1 描述性统计

In [ ]:
stats_df = pd.DataFrame()

for col in numerical_cols:
    s = df[col]
    stats_df[col] = {
        'Mean': s.mean(),
        'Median': s.median(),
        'Mode': s.mode()[0],
        'Std Dev': s.std(),
        'Variance': s.var(),
        'Range': s.max() - s.min(),
        'IQR': s.quantile(0.75) - s.quantile(0.25),
        'Skewness': s.skew(),
        'Kurtosis': s.kurtosis(),
        'CV (%)': (s.std() / s.mean()) * 100
    }

print(stats_df.round(3).to_string())

![图表：07-correlation-matrix.png](images/07-correlation-matrix.png)

描述统计从均值、中位数、离散程度、偏度和峰度等角度概括数值变量。偏度和峰度只能作为分布形态的摘要，不能单独证明数据服从或不服从正态分布；变异系数可用于比较不同量纲变量的相对波动。

#### 2.7.2 异常值检查

In [ ]:
print("异常值分析")
print("=" * 70)

outlier_summary = []

for col in numerical_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    iqr_outliers = ((df[col] < lower) | (df[col] > upper)).sum()

    z_scores = np.abs(stats.zscore(df[col]))
    z_outliers = (z_scores > 3).sum()

    outlier_summary.append({
        'Feature': col,
        'IQR Lower': round(lower, 2),
        'IQR Upper': round(upper, 2),
        'IQR Outliers': iqr_outliers,
        'IQR Outlier %': round(iqr_outliers / len(df) * 100, 2),
        'Z-score Outliers (|z|>3)': z_outliers,
        'Z-score Outlier %': round(z_outliers / len(df) * 100, 2)
    })

outlier_df = pd.DataFrame(outlier_summary)
print(outlier_df.to_string(index=False))
print()
print("Verdict: 各特征的异常值比例整体较低。")
print("本分析暂不截尾或删除记录，保留原始数据供后续判断。")

这里使用 IQR 规则和绝对 Z 分数大于 3 两种方法标记潜在异常值。它们只是筛查规则，不能自动说明记录错误。是否保留或处理异常值应结合业务含义与原始数据核查；本分析先保留全部记录。

#### 2.7.3 相关性分析

In [ ]:
corr_cols = numerical_cols + ['customer_satisfaction']
corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.3f', cmap='RdBu_r',
            center=0, square=True, linewidths=1,
            cbar_kws={'label': 'Correlation Coefficient'},
            ax=ax, vmin=-1, vmax=1)
ax.set_title('相关矩阵', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n与满意度的相关系数：")
target_corr = corr_matrix['customer_satisfaction'].drop('customer_satisfaction').sort_values(key=abs, ascending=False)
for feat, val in target_corr.items():
    strength = "较强" if abs(val) > 0.5 else "中等" if abs(val) > 0.3 else "较弱"
    print(f"  {feat}: {val:.4f} ({strength})")

相关矩阵展示数值变量之间的线性相关程度。消费金额与购物车商品数可能具有较强相关性；满意度与履约时间的相关方向值得关注。相关系数只反映线性关联，不代表因果关系，也不能排除其他变量的影响。

#### 2.7.4 统计检验

In [ ]:
print("统计检验")
print("=" * 70)

print("\n1. 正态性检验（从样本中随机抽取 5000 条进行 Shapiro–Wilk 检验）")
print("-" * 50)
np.random.seed(42)
sample_idx = np.random.choice(len(df), 5000, replace=False)

for col in numerical_cols:
    stat, p = shapiro(df[col].iloc[sample_idx])
    normal = "Yes" if p > 0.05 else "No"
    print(f"  {col}: W={stat:.4f}, p={p:.2e} -> Normal? {normal}")

print("\n多数变量的检验 p 值小于 0.05。大样本检验较敏感，建议结合图形判断分布形态，")
print("并采用对分布假设要求较少的非参数方法作组间比较。")

In [ ]:
print('Kruskal–Wallis 检验（非参数多组比较）')
print('H0：不同满意度组的该数值特征分布相同')
print('-' * 50)

for col in numerical_cols:
    groups = [group[col].values for _, group in df.groupby('customer_satisfaction')]
    stat, p = kruskal(*groups)
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    print(f'  {col}: H={stat:.2f}, p={p:.2e} {sig}')

In [ ]:
print('卡方独立性检验（类别特征 × 满意度）')
print('H0：类别特征与满意度相互独立')
print('-' * 50)

for col in cat_cols + ['is_rewards_member', 'has_food_item', 'order_ahead']:
    ct = pd.crosstab(df[col], df['customer_satisfaction'])
    chi2, p, dof, expected = chi2_contingency(ct)
    cramers_v = np.sqrt(chi2 / (len(df) * (min(ct.shape) - 1)))
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    print(f'  {col}: chi2={chi2:.1f}, p={p:.2e}, Cramér V={cramers_v:.4f} {sig}')

统计检验为图表观察提供补充：Kruskal–Wallis 检验比较不同满意度组的数值特征分布；卡方检验评估类别特征与满意度是否独立，Cramér's V 用于描述关联强度。大样本下 p 值容易显著，因此应同时看效应大小，避免只按 p 值排序。Shapiro–Wilk 检验也会对大样本中的轻微偏离敏感。

## 小结

本 Notebook 以探索性分析为主：先检查数据质量和变量分布，再比较不同满意度下的数值、类别与布尔特征，最后通过描述统计、相关分析和检验作补充。解读结果时应关注差异方向、幅度和效应大小，不要只凭 p 值下结论；这些观察也不能单独证明某项服务因素导致满意度变化。